# Lab (completed): Three ways to find a square root

> **Instructor reference copy.** This is the completed version of the lab with
> all algorithm slots filled in and example reflection answers in place. Use it
> for: (a) end-to-end testing of the notebook, (b) calibrating what a thoughtful
> student response looks like for grading, (c) live demo on a screen if a class
> gets stuck. The student-facing copy lives next to it as
> `csc113_lab_three_ways_to_find_sqrt.ipynb`.

**Module:** Planning to solve a problem  
**Estimated time:** about 45 minutes for a student

## The shape

Every algorithm in this lab fits the same template:

```
[ setup state ]
while not close enough:
    [ update state ]
return result
```

Each function also keeps a `history` list — every guess gets appended to it. The history list is *not part of the algorithm*; it's a "tap" so the plotting helpers can show where the algorithm chose to look.


In [ ]:
# Run this cell first.
import math
import matplotlib.pyplot as plt


### Provided helper: number-line plot

Students don't need to read or modify this cell — they just run it once and move on.


In [ ]:
def plot_guesses(history, target, title=""):
    """Plot every value the algorithm tested on a number line.

    Color goes from purple (early guesses) to yellow (later guesses) so you
    can see the iteration order.
    """
    sqrt_t = math.sqrt(target)
    n = len(history)

    fig, ax = plt.subplots(figsize=(10, 2.4))

    all_x = list(history) + [0, sqrt_t]
    pad = max((max(all_x) - min(all_x)) * 0.05, 0.2)
    x_min, x_max = min(all_x) - pad, max(all_x) + pad

    ax.axhline(0, color='#aaa', linewidth=0.5)
    ax.axvline(sqrt_t, color='#BA7517', linewidth=2, alpha=0.85)
    ax.text(sqrt_t, 0.32, f'sqrt({target}) approx {sqrt_t:.4f}',
            ha='center', va='bottom', fontsize=10, color='#854F0B')

    ax.scatter(history, [0] * n, c=range(n), cmap='viridis',
               s=40, alpha=0.75, zorder=3, edgecolors='none')

    if n <= 15:
        for i, x in enumerate(history):
            ax.annotate(str(i), (x, 0),
                        textcoords="offset points", xytext=(0, -15),
                        ha='center', fontsize=9, color='#444')

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(-0.6, 0.6)
    ax.set_yticks([])
    ax.set_title(f'{title} — {n} guesses', fontsize=11, loc='left', pad=10)
    for spine in ['left', 'right', 'top']:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()


def plot_comparison(histories, labels, target):
    """Stack three number lines on a shared axis for side-by-side comparison."""
    sqrt_t = math.sqrt(target)
    fig, axes = plt.subplots(len(histories), 1,
                             figsize=(10, 1.6 * len(histories)), sharex=True)
    if len(histories) == 1:
        axes = [axes]

    all_x = [x for h in histories for x in h] + [0, sqrt_t]
    pad = max((max(all_x) - min(all_x)) * 0.05, 0.2)
    x_min, x_max = min(all_x) - pad, max(all_x) + pad

    for ax, history, label in zip(axes, histories, labels):
        n = len(history)
        ax.axhline(0, color='#aaa', linewidth=0.5)
        ax.axvline(sqrt_t, color='#BA7517', linewidth=1.5, alpha=0.7)
        ax.scatter(history, [0] * n, c=range(n), cmap='viridis',
                   s=30, alpha=0.7, edgecolors='none')
        ax.set_ylim(-0.4, 0.4)
        ax.set_yticks([])
        ax.set_title(f'{label} — {n} guesses', fontsize=10, loc='left')
        for spine in ['left', 'right', 'top']:
            ax.spines[spine].set_visible(False)

    axes[-1].set_xlabel(
        f'value (sqrt({target}) approx {sqrt_t:.4f} marked in gold)', fontsize=10)
    plt.xlim(x_min, x_max)
    plt.tight_layout()
    plt.show()


## Algorithm 1: Exhaustive enumeration

Try every value, one tiny step at a time. Needs the least knowledge — only that you can square a number and check whether you're close. Does the most work.


In [ ]:
def sqrt_exhaustive(target, epsilon=0.01, step=0.001):
    """Find sqrt(target) by trying every value from 0 upward in tiny steps.

    Returns:
        (guess, iterations, history)
    """
    # Start from zero — exhaustive has no special starting point.
    guess = 0.0

    iterations = 0
    history = [guess]   # for visualization only; not part of the algorithm

    while abs(guess ** 2 - target) >= epsilon:
        # Take one small step forward and try again.
        guess += step
        iterations += 1
        history.append(guess)   # for visualization only

    return guess, iterations, history


In [ ]:
# Tests + visualization.

for t in (4, 2, 25):
    guess, iters, _ = sqrt_exhaustive(t)
    assert abs(guess ** 2 - t) < 0.01, f"Expected guess**2 near {t}, got {guess ** 2}"
    print(f"sqrt_exhaustive({t})  -> {guess:.4f}  ({iters:,} iterations)")

_, _, history = sqrt_exhaustive(2)
plot_guesses(history, 2, "Exhaustive enumeration finding sqrt(2)")


### Reflection — exhaustive enumeration

**1. What did you ask your AI partner about for this section?**

> I asked the AI to explain what `while abs(guess**2 - target) >= epsilon:` means in plain English. The first answer used a lot of technical words, so I asked it to explain like I was new to programming. It said something like "keep going as long as the guess is not close enough to the right answer." That helped me understand the loop is just checking whether to stop.

**2. Why does this algorithm work, even though it's so slow? What's the absolute minimum the algorithm needs to know about `x ** 2`?**

> It works because we test every value, so we will eventually land on one that's close. The only thing the algorithm has to know is how to square a number and how to check if two numbers are close. It doesn't need to know anything about whether the function goes up or down, or what its shape is.

**3. Look at the plot. What does the *shape* of those dots tell you about how exhaustive enumeration "thinks" about the search?**

> The dots form a long colored line going from 0 toward the target. The algorithm doesn't "think" at all — it just walks forward and checks. The shape tells me it has no idea where the answer is until it stumbles on it.


## Algorithm 2: Bisection search

Each iteration cuts the search space in half. Needs the function to be monotonic on the range, but in return does dramatically less work than exhaustive.

Assumption for this lab: `target >= 1`.


In [ ]:
def sqrt_bisection(target, epsilon=0.01):
    """Find sqrt(target) by halving the search range each step.

    Assumes target >= 1.

    Returns:
        (guess, iterations, history)
    """
    # The answer must be between 0 and target (for target >= 1).
    # We use max(1.0, target) for the upper bound so this still works
    # if someone later passes a target between 0 and 1.
    low = 0.0
    high = max(1.0, target)
    guess = (low + high) / 2

    iterations = 0
    history = [guess]   # for visualization only

    while abs(guess ** 2 - target) >= epsilon:
        # If guess squared is too small, the answer is in the upper half.
        # If it's too big, the answer is in the lower half.
        if guess ** 2 < target:
            low = guess
        else:
            high = guess
        guess = (low + high) / 2
        iterations += 1
        history.append(guess)   # for visualization only

    return guess, iterations, history


In [ ]:
# Tests + visualization.

for t in (4, 2, 25):
    guess, iters, _ = sqrt_bisection(t)
    assert abs(guess ** 2 - t) < 0.01, f"Expected guess**2 near {t}, got {guess ** 2}"
    print(f"sqrt_bisection({t})  -> {guess:.4f}  ({iters} iterations)")

_, _, history = sqrt_bisection(2)
plot_guesses(history, 2, "Bisection search finding sqrt(2)")


### Reflection — bisection search

**1. What did you ask your AI partner about for this section?**

> I asked the AI how to know which side of the range to throw away when the guess is wrong. It explained that if my guess squared is too small, the real answer has to be bigger, so I keep the bigger half. I had to push back once when it tried to just give me the code — I asked it to explain the idea instead so I could write it myself.

**2. Bisection is much faster than exhaustive. What extra piece of knowledge does it use, and why does knowing that let it skip work?**

> Bisection knows that `x ** 2` gets bigger as `x` gets bigger. That means if my guess gives a value that's too small, I know the right answer is to the right of where I looked, and I can throw away everything on the left. Exhaustive doesn't use this — it just walks forward and checks each value one at a time. The extra knowledge is what lets bisection skip past whole sections of the number line.

**3. Look at the numbered dots on the plot. Describe the pattern you see — and explain in your own words why bisection produces that pattern.**

> The dots zigzag. The first guess is at 1.0, then the next one jumps to 1.5, then back to 1.25, then 1.375, and so on. Each new dot is closer to the target than the one before, and the dots get more packed together near the answer. This happens because each step cuts the remaining range in half, so the steps get smaller as we close in.


## Algorithm 3: Newton-Raphson

Uses the slope of the function at the current guess to predict where the answer should be. Needs the most knowledge (you have to know the derivative), but converges fastest.


In [ ]:
def sqrt_newton(target, epsilon=0.01):
    """Find sqrt(target) using the Newton-Raphson update.

    Returns:
        (guess, iterations, history)
    """
    # target / 2 is a fine starting point for any positive target.
    # (We just need any nonzero positive number; if guess were 0 the update
    # would divide by zero.)
    guess = target / 2 if target > 0 else 1.0

    iterations = 0
    history = [guess]   # for visualization only

    while abs(guess ** 2 - target) >= epsilon:
        # Newton-Raphson update: jump along the tangent line to where it crosses zero.
        # x_new = x - (x**2 - target) / (2 * x)
        guess = guess - (guess ** 2 - target) / (2 * guess)
        iterations += 1
        history.append(guess)   # for visualization only

    return guess, iterations, history


In [ ]:
# Tests + visualization.

for t in (4, 2, 25):
    guess, iters, _ = sqrt_newton(t)
    assert abs(guess ** 2 - t) < 0.01, f"Expected guess**2 near {t}, got {guess ** 2}"
    print(f"sqrt_newton({t})  -> {guess:.4f}  ({iters} iterations)")

_, _, history = sqrt_newton(2)
plot_guesses(history, 2, "Newton-Raphson finding sqrt(2)")


### Reflection — Newton-Raphson

**1. What did you ask your AI partner about for this section?**

> I asked the AI to explain why the formula works without using calculus jargon. It said: imagine standing on a curve and drawing a straight line going in the direction the curve is pointing. That line crosses the target somewhere, and where it crosses is your next guess. After hearing it that way the formula stopped feeling random.

**2. Look at the Newton plot. Compared to the bisection plot, the dots don't zigzag in the same way. What does the pattern of the Newton dots tell you about how this algorithm is "thinking" differently from bisection?**

> Newton doesn't really zigzag — it makes one big jump and is almost right there. Bisection narrows in by halving, but Newton is predicting. The dots are at 1.0, then 1.5, then basically on top of the target. It feels less like "search" and more like "calculation."

**3. Newton needs a reasonable starting guess. What do you think could go wrong if your starting guess were really, really far from the correct answer?**

> If I started really far away, the slope at that point might point in a weird direction, so the "predicted" next guess could overshoot wildly. It might even bounce back and forth without getting closer, or maybe shoot off to infinity. The further the starting point, the riskier the prediction.


## Side by side


In [ ]:
target = 2

g_e, i_e, h_e = sqrt_exhaustive(target)
g_b, i_b, h_b = sqrt_bisection(target)
g_n, i_n, h_n = sqrt_newton(target)

print(f"Finding sqrt({target}) — true value: {math.sqrt(target):.6f}")
print(f"  exhaustive enumeration  -> {g_e:.4f}  in {i_e:>6,} iterations")
print(f"  bisection search        -> {g_b:.4f}  in {i_b:>6,} iterations")
print(f"  Newton-Raphson          -> {g_n:.4f}  in {i_n:>6,} iterations")
print()

plot_comparison([h_e, h_b, h_n],
                ['Exhaustive', 'Bisection', 'Newton-Raphson'],
                target)


### Big-picture reflection

**1. In your own words, what's the connection between *how much an algorithm assumes about the problem* and *how much work it has to do*?**

> The more the algorithm assumes — or knows — about the problem, the less work it has to do. Exhaustive assumes almost nothing and walks step by step. Bisection assumes the function goes up, and uses that to skip half the search every step. Newton assumes you know the slope, and uses that to predict where the answer is. Each step up the ladder buys you a lot of speed.

**2. Where in this lab did your AI partner help you the most? Where did it help the least?**

> The AI was best at explaining concepts in plain language — like what monotonic means, or what Newton's formula is actually doing. It was weakest when I made typos. A couple of times instead of pointing at the typo it rewrote my whole function "the right way," which made it harder for me to learn. I learned to ask for hints instead of code.

**3. Where else have you seen this pattern — a fixed outer routine with a varying inner step?**

> Cooking. You taste a sauce, decide if it needs something, add a pinch, taste again, repeat until it's right. The check is the loop condition, the add is the update. I think studying for a test is similar — read, quiz yourself, see what you got wrong, study that, quiz again.


## Submission

In the student version: Restart & Run All, save, commit, push. Standard week-1 submission flow.

This completed copy is for instructor use — not for student submission.


### Stuck-points

> (Empty in the completed reference copy. In a real student submission this is where they describe what they got stuck on, for partial credit.)
